# 02 — Nettoyage et construction des indicateurs

Objectif : passer du format long (une ligne par agrégat) au format large (une ligne par commune/année), filtrer sur le périmètre d'étude, et construire les indicateurs financiers manquants (taux d'endettement, ratio investissement).

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
from utils import load_raw_data, clean_columns, pivot_agregats, compute_taux_endettement

In [2]:
df = load_raw_data('../data/raw/communes.csv')
df.shape

(2755101, 42)

## Filtrage sur le budget principal

Le dataset contient budgets principaux et annexes. On ne garde que le budget principal pour ne pas fausser les agrégats globaux.

In [3]:
df['Type de budget'].value_counts()

Budget principal    1733849
Budget annexe       1021252
Name: Type de budget, dtype: int64

In [4]:
df_bp = df[df['Type de budget'] == 'Budget principal'].copy()
df_bp.shape

(1733849, 42)

## Pivot : format long -> format large

Une ligne par commune/année, une colonne par agrégat (Epargne brute, Encours de dette, etc.).

In [16]:
pivot = pivot_agregats(df_bp)
pivot.shape

(35557, 59)

In [7]:
pivot.head()

,Exercice,Code Insee 2025 Commune,Nom 2025 Commune,Nom 2025 Département,Strate population 2025,Achats et charges externes,Annuité de la dette,Autres dotations de fonctionnement,Autres dotations et subventions,Autres dépenses d'investissement,...,Recettes totales hors emprunts,Remboursements d'emprunts hors GAD,Subventions aux personnes de droit privé,Subventions d'équipement versées,Subventions reçues et participations,TVA,Taxe d'enlévement des ordures ménagères,Variation du fonds de roulement,Ventes de biens et services,Versement transport
0,2018,9001,Aigues-Juntes,Ariège,0,11705.74,0.00,2972.00,0.00,0.0,...,49188.81,0.00,510.0,0.00,0.0,NaN,NaN,9984.64,203.00,NaN
1,2018,9002,Aigues-Vives,Ariège,3,97977.93,28414.31,3011.73,64354.08,0.0,...,409115.48,15453.66,NaN,0.00,8773.9,NaN,NaN,190118.36,33928.60,NaN
2,2018,9003,L'Aiguillon,Ariège,2,50808.07,18150.14,2972.00,78951.44,0.0,...,261322.61,15635.29,1200.0,0.00,0.0,NaN,NaN,20669.02,883.00,NaN
3,2018,9004,Albiès,Ariège,1,55463.12,12402.95,NaN,795.40,0.0,...,166728.34,11198.22,6135.0,9743.62,5853.0,NaN,587.0,-10207.13,18923.85,NaN
4,2018,9005,Aleu,Ariège,1,48196.37,13247.85,2972.00,22173.00,0.0,...,225819.04,10870.12,1600.0,0.00,9870.1,NaN,NaN,22309.04,17432.67,NaN


In [9]:
pivot['Nom 2025 Commune'].nunique()

4242

In [ ]:
#Vérification qu'il n'y ait pas de doublon, si le code renvoie 1 c'est qu'il n'y a pas de doublon
pivot.groupby('Code Insee 2025 Commune')['Nom 2025 Département'].nunique().max()

1

In [ ]:
#Toutefois on s'aperçoit qu'il y a des homonymes
communes_multi_dept = pivot.groupby('Nom 2025 Commune')['Nom 2025 Département'].nunique()
communes_multi_dept[communes_multi_dept > 1]

Nom 2025 Commune
Aigues-Vives    4
Albas           2
Albiac          2
Alos            2
Antras          2
               ..
Verfeil         2
Viella          2
Villefort       2
Villeneuve      2
Vira            2
Name: Nom 2025 Département, Length: 165, dtype: int64

## Sélection des colonnes utiles

On garde uniquement ce qui sert à répondre aux 4 questions métier du README.

In [17]:
colonnes_utiles = [
    'Exercice', 'Code Insee 2025 Commune', 'Nom 2025 Commune', 'Nom 2025 Département', 'Strate population 2025',
    'Epargne brute', 'Encours de dette', 'Recettes de fonctionnement',
    "Dépenses d'investissement", 'Recettes totales'
]
pivot = pivot[colonnes_utiles]

## Construction des indicateurs manquants

In [20]:
pivot = compute_taux_endettement(pivot)
pivot['ratio_investissement'] = pivot["Dépenses d'investissement"] / pivot['Recettes totales']
pivot.head()

,Exercice,Code Insee 2025 Commune,Nom 2025 Commune,Nom 2025 Département,Strate population 2025,Epargne brute,Encours de dette,Recettes de fonctionnement,Dépenses d'investissement,Recettes totales,taux_endettement,ratio_investissement
0,2018,9001,Aigues-Juntes,Ariège,0,26044.43,0.00,48073.82,17174.78,49188.81,0.000000,0.349160
1,2018,9002,Aigues-Vives,Ariège,3,83889.75,752163.15,340358.38,329318.49,775905.48,2.209915,0.424431
2,2018,9003,L'Aiguillon,Ariège,2,55448.87,158673.24,175253.30,120849.16,261322.61,0.905394,0.462452
3,2018,9004,Albiès,Ariège,1,28044.88,20638.49,165632.94,49091.03,176471.96,0.124604,0.278180
4,2018,9005,Aleu,Ariège,1,28469.39,215236.68,191581.99,40397.40,225819.04,1.123470,0.178893


## Valeurs manquantes et doublons

In [21]:
pivot.isna().mean()

Exercice                      0.0
Code Insee 2025 Commune       0.0
Nom 2025 Commune              0.0
Nom 2025 Département          0.0
Strate population 2025        0.0
Epargne brute                 0.0
Encours de dette              0.0
Recettes de fonctionnement    0.0
Dépenses d'investissement     0.0
Recettes totales              0.0
taux_endettement              0.0
ratio_investissement          0.0
dtype: float64

In [22]:
print('Doublons :', pivot.duplicated().sum())

Doublons : 0


## Décision sur les valeurs manquantes

Valeurs manquantes : 0% sur toutes les colonnes, le dataset est bien complet sur ce périmètre, pas de dropna() à faire

Vérification des doublons : Aucun doublon non plus, quelques homonymes mais pas dérangeant en se basant sur le Code Insee comme index.

In [ ]:
# pivot = pivot.dropna(subset=['Epargne brute', 'Encours de dette', 'Recettes de fonctionnement'])
# pivot.shape

## Sauvegarde du dataset nettoyé

In [24]:
pivot.to_csv('../data/raw/communes_occitanie_clean.csv', index=False)